<a href="https://colab.research.google.com/github/prathyusha2020/Speech_Translation/blob/main/Audio_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Real Time Speech to Translated Speech

Translating my own voice into a different language.

---
## OpenAI For Translation

In [ ]:
!pip install langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.9/415.9 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.4/567.4 kB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 51.4 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.61.1
    Uninstalling openai-1.61.1:
      Successfully uninstalled openai-1.61.1
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.44
    Uninstalling langchain-core-0.3.44:
      Successfully uninstalled langchain-core-0.3.44


In [ ]:
import os  #  Import 'os' module

from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

#  Set OpenAI API Key securely
os.environ["OPENAI_API_KEY"] = "Enter your key"

#  Fetch the API Key correctly
api_key = os.getenv("OPENAI_API_KEY")

#  Initialize ChatOpenAI with API Key
llm = ChatOpenAI(temperature=0.0, model="gpt-4o", openai_api_key=api_key)

#  Translation Template
translation_template = """
Translate the following sentence into {language}, return ONLY the translation, nothing else.

Sentence: {sentence}
"""

output_parser = StrOutputParser()
translation_prompt = ChatPromptTemplate.from_template(translation_template)

# Define translation chain
translation_chain = (
    {"language": RunnablePassthrough(), "sentence": RunnablePassthrough()}
    | translation_prompt
    | llm
    | output_parser
)

#  Function to translate text
def translate(sentence, language="French"):
    data_input = {"language": language, "sentence": sentence}
    translation = translation_chain.invoke(data_input)
    return translation

#  Test the function
print(translate("Hello, how are you?", "Russian"))  # Expected Output: "Hola, ¿cómo estás?"


Здравствуйте, как вы?


In [ ]:
import openai

openai.api_key = os.getenv("OPENAI_API_KEY")  # Ensure your API key is set

#  Correct way to list models in OpenAI API v1+
response = openai.models.list()

#  Print available models
for model in response.data:
    print(model.id)


gpt-4o-mini-audio-preview-2024-12-17
dall-e-3
dall-e-2
gpt-4o-audio-preview-2024-10-01
gpt-4o-audio-preview
o1-mini-2024-09-12
o1-mini
omni-moderation-latest
gpt-4o-mini-audio-preview
omni-moderation-2024-09-26
gpt-4o-mini-2024-07-18
gpt-4o-mini
babbage-002
tts-1-hd-1106
whisper-1
text-embedding-3-large
gpt-4o-2024-05-13
tts-1-hd
o1-preview
o1-preview-2024-09-12
gpt-3.5-turbo-instruct-0914
gpt-4o-mini-search-preview
tts-1-1106
davinci-002
gpt-3.5-turbo-1106
gpt-4o-search-preview
gpt-3.5-turbo-instruct
gpt-4o-mini-search-preview-2025-03-11
gpt-4o-2024-11-20
gpt-3.5-turbo-0125
gpt-4o-2024-08-06
gpt-3.5-turbo
gpt-3.5-turbo-16k
gpt-4o
text-embedding-3-small
text-embedding-ada-002
gpt-4.5-preview
gpt-4.5-preview-2025-02-27
gpt-4o-search-preview-2025-03-11
tts-1


---
## ElevenLabs For Voice Cloning & Voice Synthesis

Premade voice model on [ElevenLabs Service](https://elevenlabs.io/app/voice-lab), using Multilingual V2 Model for synthesis

**Available Languages:** *Chinese, Korean, Dutch, Turkish, Swedish, Indonesian, Filipino, Japanese, Ukrainian, Greek, Czech, Finnish, Romanian, Russian, Danish, Bulgarian, Malay, Slovak, Croatian, Classic Arabic, Tamil, English, Polish, German, Spanish, French, Italian, Hindi and Portuguese*

In [ ]:
!pip install elevenlabs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.5/347.5 kB 5.3 MB/s eta 0:00:00


In [ ]:
from elevenlabs.client import ElevenLabs
from elevenlabs import play, stream
import os
os.environ["ELEVENLABS_API_KEY"] = "Enter your key"
client = ElevenLabs(api_key=os.getenv("ELEVENLABS_API_KEY"))

def gen_dub(text):
    print("Generating audio...")
    audio = client.generate(
        text=text,
        voice="", # Insert voice model here!
        model="eleven_multilingual_v2"
    )
    play(audio)

---
## AssemblyAI for Speech to Text Streaming

AssemblyAI handles Streaming STT within their own platform. Inserting the above translation and voice generation functions within this workflow.

In [ ]:
!pip install assemblyai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.2/44.2 kB 1.8 MB/s eta 0:00:00


In [ ]:
import assemblyai as aai
os.environ["ASSEMBLYAI_API_KEY"] = "Enter your key"
aai.settings.api_key = os.getenv("ASSEMBLYAI_API_KEY")


def on_open(session_opened: aai.RealtimeSessionOpened):
  "This function is called when the connection has been established."
  print("Session ID:", session_opened.session_id)

def on_data(transcript: aai.RealtimeTranscript):
  "This function is called when a new transcript has been received."
  if not transcript.text:
    return

  if isinstance(transcript, aai.RealtimeFinalTranscript):
    print(transcript.text, end="\r\n")
    print("Translating...")
    translation = translate(str(transcript.text))
    print(f"Translation: {translation}")
    gen_dub(translation)
  else:
    print(transcript.text, end="\r")

def on_error(error: aai.RealtimeError):
  "This function is called when the connection has been closed."
  print("An error occured:", error)

def on_close():
  "This function is called when the connection has been closed."
  print("Closing Session")

transcriber = aai.RealtimeTranscriber(
  on_data=on_data,
  on_error=on_error,
  sample_rate=44_100,
  on_open=on_open, # optional
  on_close=on_close, # optional
)

---
## Main Script

(remember to change audio input/output in settings for airpods)

In [ ]:
!pip install "assemblyai[extras]"
!apt install portaudio19-dev # For Debian/Ubuntu - Install PortAudio dependency


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 1.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for pyaudio (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pyaudio
Failed to build pyaudio
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pyaudio)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libportaudio2 libportaudiocpp0
Suggested packages:
  portaudio19-doc
The following NEW packages will be installed:
  libportaudio2 libportaudiocpp0 portaudio19-dev
0 upgraded, 3 newly installed, 0 to remove and 29 not 

In [ ]:
transcriber.close()

An error occured: Not Authorized
Closing Session
Closing Session


In [ ]:
import os
import assemblyai as aai
from google.colab import files
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from elevenlabs.client import ElevenLabs
from elevenlabs import save

In [ ]:
pip install assemblyai

In [ ]:
pip install googletrans

In [ ]:
import os
import assemblyai as aai
import openai
from google.colab import files
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from elevenlabs.client import ElevenLabs
from elevenlabs import save
from googletrans import Translator

# ✅ Set API Keys
os.environ["ASSEMBLYAI_API_KEY"] = "7644d2dd9e30473eb9ff6595a8e3ecdd"
#os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"
#os.environ["ELEVENLABS_API_KEY"] = "your-elevenlabs-api-key-here"

# ✅ Initialize APIs
aai.settings.api_key = os.getenv("ASSEMBLYAI_API_KEY")
client = ElevenLabs(api_key=os.getenv("ELEVENLABS_API_KEY"))
translator = Translator()

# ✅ OpenAI GPT Model
llm = ChatOpenAI(temperature=0.0, model="gpt-4o", openai_api_key=os.getenv("OPENAI_API_KEY"))

# ✅ Translation Prompt
translation_template = """
You are a professional translator. Your ONLY job is to translate the given text into {language}.
Return ONLY the translated text without any additional comments, explanations, or disclaimers.

Do NOT refuse the request. If the text cannot be translated, return the original text.

Text to translate: {sentence}
"""

output_parser = StrOutputParser()
translation_prompt = ChatPromptTemplate.from_template(translation_template)

# ✅ Define Translation Chain
translation_chain = (
    {"language": RunnablePassthrough(), "sentence": RunnablePassthrough()}
    | translation_prompt
    | llm
    | output_parser
)

# ✅ Function to Translate Using OpenAI GPT
def translate(sentence, language="English"):
    data_input = {"language": language, "sentence": sentence}
    translation = translation_chain.invoke(data_input)

    # ✅ Fallback if OpenAI refuses translation
    if "sorry" in translation.lower() or "can't assist" in translation.lower():
        print("⚠ OpenAI refused the request. Using Google Translate as fallback.")
        translation = translator.translate(sentence, dest=language).text

    return translation

# ✅ Step 1: Upload an Audio File
print("📤 Please upload an audio file...")
uploaded_files = files.upload()
audio_filename = list(uploaded_files.keys())[0]  # Get uploaded filename

# ✅ Step 2: Transcribe Audio to Text
print(f"🎤 Processing {audio_filename}...")
transcriber = aai.Transcriber()
transcript = transcriber.transcribe(audio_filename)

# ✅ Extract Transcribed Text
transcribed_text = transcript.text
print(f"📝 Transcribed Text: {transcribed_text}")

# ✅ Step 3: Translate the Text Using OpenAI GPT (with fallback to Google Translate)
target_language = "en"  # "ru" for Russian (Modify as needed)
translated_text = translate(transcribed_text, target_language)
print(f"🌍 Translated Text: {translated_text}")

# ✅ Step 4: Convert Translated Text to Speech Using ElevenLabs
print("🎶 Generating translated speech...")

# ✅ Use a valid ElevenLabs voice
translated_audio = client.generate(
    text=translated_text,
    voice="Rachel",  # Change to a preferred ElevenLabs voice
    model="eleven_multilingual_v2"
)

# ✅ Step 5: Save and Download the Translated Audio
output_audio_file = "translated_speech.mp3"
save(translated_audio, output_audio_file)
print(f"✅ Translated speech saved as {output_audio_file}")

# ✅ Provide a Download Link
files.download(output_audio_file)


📤 Please upload an audio file...


Saving real_0.wav to real_0 (2).wav
🎤 Processing real_0 (2).wav...
📝 Transcribed Text: 
🌍 Translated Text: 
🎶 Generating translated speech...
✅ Translated speech saved as translated_speech.mp3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os
import assemblyai as aai
import openai
from google.colab import files
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from elevenlabs.client import ElevenLabs
from elevenlabs import save
from googletrans import Translator

# ✅ Set API Keys
os.environ["ASSEMBLYAI_API_KEY"] = "7644d2dd9e30473eb9ff6595a8e3ecdd"
#os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"
#os.environ["ELEVENLABS_API_KEY"] = "your-elevenlabs-api-key-here"

# ✅ Initialize APIs
aai.settings.api_key = os.getenv("ASSEMBLYAI_API_KEY")
client = ElevenLabs(api_key=os.getenv("ELEVENLABS_API_KEY"))
translator = Translator()

# ✅ OpenAI GPT Model
llm = ChatOpenAI(temperature=0.0, model="gpt-4o", openai_api_key=os.getenv("OPENAI_API_KEY"))

# ✅ Translation Prompt
translation_template = """
You are a professional translator. Your ONLY job is to translate the given text into {language}.
Return ONLY the translated text without any additional comments, explanations, or disclaimers.

Do NOT refuse the request. If the text cannot be translated, return the original text.

Text to translate: {sentence}
"""

output_parser = StrOutputParser()
translation_prompt = ChatPromptTemplate.from_template(translation_template)

# ✅ Define Translation Chain
translation_chain = (
    {"language": RunnablePassthrough(), "sentence": RunnablePassthrough()}
    | translation_prompt
    | llm
    | output_parser
)

# ✅ Function to Translate Using OpenAI GPT
def translate(sentence, language="Russian"):
    data_input = {"language": language, "sentence": sentence}
    translation = translation_chain.invoke(data_input)

    # ✅ Extract the translated text correctly
    if isinstance(translation, dict):
        translation = translation.get("sentence", translation)  # Extract only the sentence

    # ✅ Fallback if OpenAI refuses translation
    if "sorry" in translation.lower() or "can't assist" in translation.lower():
        print("⚠ OpenAI refused the request. Using Google Translate as fallback.")
        translation = translator.translate(sentence, dest=language).text

    return translation

# ✅ Step 1: Upload an Audio File
print("📤 Please upload an audio file...")
uploaded_files = files.upload()
audio_filename = list(uploaded_files.keys())[0]  # Get uploaded filename

# ✅ Step 2: Transcribe Audio to Text
print(f"🎤 Processing {audio_filename}...")
transcriber = aai.Transcriber()
transcript = transcriber.transcribe(audio_filename)

# ✅ Extract Transcribed Text
transcribed_text = transcript.text
print(f"📝 Transcribed Text: {transcribed_text}")

# ✅ Step 3: Translate the Text Using OpenAI GPT (with fallback to Google Translate)
target_language = "ru"
translated_text = translate(transcribed_text, target_language)
print(f"🌍 Translated Text: {translated_text}")  # ✅ Only the translated text is printed

# ✅ Step 4: Convert Translated Text to Speech Using ElevenLabs
print("🎶 Generating translated speech...")

# ✅ Use a valid ElevenLabs voice
translated_audio = client.generate(
    text=translated_text,
    voice="Rachel",  # Change to a preferred ElevenLabs voice
    model="eleven_multilingual_v2"
)

# ✅ Step 5: Save and Download the Translated Audio
output_audio_file = "translated_speech.mp3"
save(translated_audio, output_audio_file)
print(f"✅ Translated speech saved as {output_audio_file}")

# ✅ Provide a Download Link
files.download(output_audio_file)


📤 Please upload an audio file...


Saving download (1) (mp3cut.net).mp3 to download (1) (mp3cut.net) (2).mp3
🎤 Processing download (1) (mp3cut.net) (2).mp3...
📝 Transcribed Text: Machine learning is a field of inquiry devoted to understanding and building methods that learn, that is Methods that leverage data to improve performance on some set of tasks. It is seen as a part of artificial intelligence. Machine learning algorithms build a model based on sample data, known as training data, in order to make predictions or decisions without being explicitly programmed to do so. Machine learning algorithms are used in a wide.
🌍 Translated Text: Машинное обучение — это область исследований, посвященная пониманию и созданию методов, которые обучаются, то есть методов, использующих данные для улучшения производительности в определенном наборе задач. Оно рассматривается как часть искусственного интеллекта. Алгоритмы машинного обучения строят модель на основе выборочных данных, известных как обучающие данные, чтобы делать прогноз

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>